### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002084D523CB0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002084D738AD0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="the director of the movie")
    ratings:float=Field(description="The movie's ratings out of 10")

In [6]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002084D523CB0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002084D738AD0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The ti

In [9]:
response = model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', ratings=8.8)

### Message output and parsed Structured Output

In [12]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(...,description="The title of the movie")
    year: int = Field(...,description="This year the movie was released")
    director: str = Field(...,description="the director of the movie")
    ratings: float = Field(...,description="The movie's ratings out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants details about the movie Inception. We can use the provided function Movie to get details. Need director, ratings, title, year. Provide those. Use function.', 'tool_calls': [{'id': 'fc_a68f82e2-4e87-4b4a-9a3f-8083ff850d80', 'function': {'arguments': '{"director":"Christopher Nolan","ratings":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 159, 'total_tokens': 248, 'completion_time': 0.189941792, 'completion_tokens_details': {'reasoning_tokens': 37}, 'prompt_time': 0.005979691, 'prompt_tokens_details': None, 'queue_time': 0.347252104, 'total_time': 0.195921483}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_fe269835c7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0cd42-dd04-7f32-8f3c-d5f2f1d5341d-0', tool

### Nested Structure

In [19]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str = Field(..., description="The name of the actor")
    role: str = Field(..., description="The role of the actor in the movie")

class MovieDetails(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="This year the movie was released")
    cast: list[Actor] = Field(..., description="The cast of the movie")
    genres: list[str] = Field(..., description="The genre of the movie")
    budget: float | None = Field(None, description="budget in Million USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Michael Caine', role='Professor Stephen Miles')], genres=['Action', 'Adventure', 'Sci-Fi'], budget=160000000.0)

### TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [21]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie out of 10"]    

model_with_typedict = model.with_structured_output(MovieDict)
response = model_with_typedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [22]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("provide details about the movie avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Clark Gregg', 'role': 'Phil Coulson'},
  {'name': 'Cobie Smulders', 'role': 'Maria Hill'},
  {'name': 'Stellan Skarsgård', 'role': 'Erik Selvig'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi', 'Superhero'],
 'title': 'The Avengers',
 'year': 2012}

In [24]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses

a data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator


In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model = "groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role":"user", "content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result


{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='d449206b-afe7-4481-80a7-ed70af6194dd'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'The user asks to extract contact info from a string. We need to output JSON matching the ContactInfo schema: fields name, email, phone, all required. Must be compact JSON. Provide only final JSON object.\n\nExtracted: name "John Doe", email "john@example.com", phone "(555) 123-4567". Ensure JSON is compact, no spaces? Compact means minimal spaces. Provide {"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}.\n\nCheck schema: order not required. All required present. Output only JSON.'}, response_metadata={'token_usage': {'completion_tokens': 153, 'prompt_tokens': 238, 'total_tokens': 391, 'completion_time': 0.326477305, 'completion_tokens_de

In [30]:
print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [31]:
## TypeDict imple

from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact info of a person"""
    name: str
    email: str
    phone: str

agent = create_agent(
    model = "groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role":"user", "content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result


{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='03547d16-5713-4f5d-80da-b47cabdb5300'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'We need to output JSON conforming to ContactInfo schema: fields name, email, phone all required. Provide compact JSON. So output: {"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"} Ensure no extra whitespace? Compact means minimal spaces. We\'ll output exactly that.'}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 212, 'total_tokens': 315, 'completion_time': 0.216496264, 'completion_tokens_details': {'reasoning_tokens': 67}, 'prompt_time': 0.010075389, 'prompt_tokens_details': None, 'queue_time': 0.256850023, 'total_time': 0.226571653}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_27194a498

In [43]:
print(result['structured_response'])

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


In [47]:
## DataClass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact info for a person"""
    name: str
    email: str
    phone: str

agent = create_agent(
    model = "groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{
    "role":"user",
    "content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"
}]
})

result
result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')